# Mid-Fusion Multimodal Model (Image + Report Text)

This notebook demonstrates a mid-fusion architecture for the Kaggle Knee MRI challenge that combines:
- **Image Encoder**: 3D CNN processing MRI volume patches
- **Text Encoder**: Transformer-based encoder for radiologist reports
- **Fusion Layer**: Late concatenation of image and text embeddings
- **Classification Head**: Multi-label predictions for abnormality detection

## Key Features
- Integrates with preprocessed `.npz` files (from `preprocess.py`)
- Compatible with existing PyTorch Lightning training pipeline
- Validates dataset loading and batch formatting
- End-to-end training with 5-fold cross-validation support

## 1. Setup and Environment

In [ ]:
import sys
import os
from pathlib import Path

# Add project root to path
project_root = Path('../../').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger
import timm
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, Dataset

# Set seeds for reproducibility
pl.seed_everything(42)
torch.set_float32_matmul_precision('high')

print(f'PyTorch: {torch.__version__}')
print(f'PyTorch Lightning: {pl.__version__}')
print(f'GPU Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Custom Modules for Mid-Fusion Architecture

In [ ]:
class AttentionPool(nn.Module):
    """Attention-based pooling for aggregating slice features."""
    def __init__(self, dim):
        super().__init__()
        self.q = nn.Linear(dim, 1)

    def forward(self, x):  # x: (B, S, D)
        w = torch.softmax(self.q(x).squeeze(-1), dim=1)  # (B, S)
        return (x * w.unsqueeze(-1)).sum(1)  # (B, D)


class ImageEncoder(nn.Module):
    """Image encoder using pretrained 2D CNN backbone (processes slices individually)."""
    def __init__(self, backbone_name='tf_efficientnet_b3', embed_dim=256):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=True, 
                                          num_classes=0, global_pool='')
        self.feat_dim = self.backbone.num_features
        self.pool = AttentionPool(self.feat_dim)
        self.proj = nn.Sequential(
            nn.Linear(self.feat_dim, embed_dim),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

    def forward(self, x):  # x: (B, S, 1, H, W)
        B, S, C, H, W = x.shape
        # Reshape to (B*S, C, H, W)
        x = x.view(B * S, C, H, W)
        # Convert grayscale to RGB if needed
        if C == 1:
            x = x.repeat(1, 3, 1, 1)
        # Extract features
        feats = self.backbone.forward_features(x)  # (B*S, feat_dim, h, w)
        # Global average pooling if needed
        if feats.dim() == 4:
            feats = torch.mean(feats, dim=[2, 3])  # (B*S, feat_dim)
        # Reshape back to (B, S, feat_dim)
        feats = feats.view(B, S, -1)
        # Attention pooling across slices
        pooled = self.pool(feats)  # (B, feat_dim)
        # Project to embedding dimension
        emb = self.proj(pooled)  # (B, embed_dim)
        return emb


class TextEncoder(nn.Module):
    """Text encoder using embedding + BiLSTM pooling."""
    def __init__(self, vocab_size=5000, embed_dim=128, hidden_dim=256, embed_out=256):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, 
                           bidirectional=True, num_layers=2, dropout=0.2)
        self.proj = nn.Sequential(
            nn.Linear(hidden_dim * 2, embed_out),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

    def forward(self, x):  # x: (B, seq_len)
        # Embed text
        emb = self.embedding(x)  # (B, seq_len, embed_dim)
        # LSTM encode
        _, (h_n, _) = self.lstm(emb)  # h_n: (2*num_layers, B, hidden_dim)
        # Concatenate last hidden states from both directions
        h_fwd = h_n[-2]  # (B, hidden_dim)
        h_bwd = h_n[-1]  # (B, hidden_dim)
        h = torch.cat([h_fwd, h_bwd], dim=1)  # (B, 2*hidden_dim)
        # Project
        emb_out = self.proj(h)  # (B, embed_out)
        return emb_out


def compute_macro_auc(y_true, y_score):
    """Compute macro-averaged AUC across all label classes."""
    try:
        n_classes = y_true.shape[1]
        aucs = []
        for i in range(n_classes):
            try:
                auc = roc_auc_score(y_true[:, i], y_score[:, i])
                aucs.append(auc)
            except:
                pass
        return float(np.mean(aucs)) if aucs else 0.0
    except:
        return 0.0

## 3. Mid-Fusion Lightning Module

In [ ]:
class MidFusionMultimodal(pl.LightningModule):
    """Mid-fusion model combining image and text modalities."""
    
    def __init__(self, n_outputs=12, image_backbone='tf_efficientnet_b3',
                 embed_dim=256, vocab_size=5000, lr=1e-4):
        super().__init__()
        self.save_hyperparameters()
        
        # Encoders
        self.image_encoder = ImageEncoder(backbone_name=image_backbone, 
                                          embed_dim=embed_dim)
        self.text_encoder = TextEncoder(vocab_size=vocab_size, 
                                       embed_out=embed_dim)
        
        # Fusion and classification head
        self.fusion_head = nn.Sequential(
            nn.Linear(embed_dim * 2, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, n_outputs)
        )
        
        self.criterion = nn.BCEWithLogitsLoss()
        
        # Validation buffers
        self.val_targets = []
        self.val_preds = []

    def forward(self, images, texts):
        """Forward pass combining image and text.
        Args:
            images: (B, S, 1, H, W)
            texts: (B, seq_len)
        Returns:
            logits: (B, n_outputs)
        """
        # Encode modalities
        img_emb = self.image_encoder(images)  # (B, embed_dim)
        txt_emb = self.text_encoder(texts)    # (B, embed_dim)
        
        # Concatenate at fusion layer (mid-fusion)
        fused = torch.cat([img_emb, txt_emb], dim=1)  # (B, 2*embed_dim)
        
        # Classification
        logits = self.fusion_head(fused)  # (B, n_outputs)
        return logits

    def training_step(self, batch, batch_idx):
        images = batch['image']
        texts = batch['text']
        labels = batch['label']
        
        logits = self(images, texts)
        loss = self.criterion(logits, labels)
        
        self.log('train/loss', loss, on_step=True, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        images = batch['image']
        texts = batch['text']
        labels = batch['label']
        
        logits = self(images, texts)
        loss = self.criterion(logits, labels)
        
        # Accumulate for macro AUC calculation
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        labels_np = labels.detach().cpu().numpy()
        self.val_preds.append(probs)
        self.val_targets.append(labels_np)
        
        self.log('val/loss', loss, on_step=False, on_epoch=True, prog_bar=True)

    def on_validation_epoch_end(self):
        if len(self.val_targets) == 0:
            return
        
        y_true = np.concatenate(self.val_targets, axis=0)
        y_score = np.concatenate(self.val_preds, axis=0)
        macro_auc = compute_macro_auc(y_true, y_score)
        
        self.log('val/macro_auc', macro_auc, prog_bar=True)
        
        # Clear buffers
        self.val_targets = []
        self.val_preds = []

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), 
                                      lr=self.hparams.lr, 
                                      weight_decay=1e-2)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=20, eta_min=1e-6
        )
        return [optimizer], [{'scheduler': scheduler, 'interval': 'epoch'}]

    def predict_step(self, batch, batch_idx):
        images = batch['image']
        texts = batch['text']
        logits = self(images, texts)
        probs = torch.sigmoid(logits)
        return probs.detach().cpu().numpy()

## 4. Multimodal Dataset Implementation

In [ ]:
class MultimodalStudyDataset(Dataset):
    """Dataset loading preprocessed MRI volumes and corresponding reports.
    
    Expected CSV columns:
        - id: study ID
        - path: path to .npz file
        - text_encoded: tokenized report (space-separated integers)
        - label_0 ... label_11: abnormality labels (0/1)
    """
    def __init__(self, rows, n_slices=16, max_text_len=256):
        self.rows = rows
        self.n_slices = n_slices
        self.max_text_len = max_text_len

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        
        # Load image volume
        npz = np.load(r['path'], allow_pickle=True)
        vol = npz['images'] if 'images' in npz else npz[npz.files[0]]
        
        # Sample slices
        Z = vol.shape[0]
        if Z >= self.n_slices:
            indices = np.linspace(0, Z - 1, self.n_slices, dtype=int)
        else:
            indices = np.concatenate([np.arange(Z), 
                                    np.repeat(Z - 1, self.n_slices - Z)])
        slices = vol[indices]  # (n_slices, H, W)
        x_img = np.expand_dims(slices, axis=1)  # (n_slices, 1, H, W)
        x_img = torch.from_numpy(x_img).float()
        
        # Load text (tokenized)
        text_str = r.get('text_encoded', '')
        if isinstance(text_str, str) and text_str.strip():
            tokens = [int(t) for t in text_str.strip().split()]
        else:
            tokens = [1]  # UNK token
        
        # Pad or truncate
        if len(tokens) < self.max_text_len:
            tokens = tokens + [0] * (self.max_text_len - len(tokens))
        else:
            tokens = tokens[:self.max_text_len]
        x_txt = torch.tensor(tokens, dtype=torch.long)
        
        # Load labels
        label_cols = [c for c in r.keys() if c.startswith('label_')]
        if label_cols:
            y = torch.tensor([float(r[c]) for c in sorted(label_cols)], 
                            dtype=torch.float32)
        else:
            y = None
        
        return {
            'image': x_img,
            'text': x_txt,
            'label': y,
            'id': r['id']
        }


def collate_multimodal(batch):
    """Collate function for multimodal batches."""
    images = torch.stack([b['image'] for b in batch], dim=0)  # (B, S, 1, H, W)
    texts = torch.stack([b['text'] for b in batch], dim=0)    # (B, seq_len)
    ids = [b['id'] for b in batch]
    labels = None
    
    if batch[0]['label'] is not None:
        labels = torch.stack([b['label'] for b in batch], dim=0)  # (B, n_outputs)
    
    return {
        'image': images,
        'text': texts,
        'label': labels,
        'id': ids
    }

## 5. Data Loading and Validation

In [ ]:
# Example: Load dataset CSV (if it exists)
csv_path = Path('challenges/kaggle-knee-mri/data/dataset.csv')

if csv_path.exists():
    df = pd.read_csv(csv_path)
    print(f"Dataset shape: {df.shape}")
    print(f"\nColumns: {list(df.columns)}")
    print(f"\nFirst few rows:")
    print(df.head())
    print(f"\nData types:\n{df.dtypes}")
    print(f"\nMissing values:\n{df.isnull().sum()}")
else:
    print(f"Dataset CSV not found at {csv_path}")
    print("To use this notebook, create a CSV with columns:")
    print("  - id: study ID")
    print("  - path: path to preprocessed .npz file")
    print("  - fold: fold number (0-4 for 5-fold CV)")
    print("  - text_encoded: tokenized report text (space-separated integers)")
    print("  - label_0 ... label_11: abnormality labels")

## 6. Validation: Dataset Format Check

In [ ]:
# Test dataset loading with a small subset
if csv_path.exists():
    df = pd.read_csv(csv_path)
    
    # Take first 5 samples
    test_rows = []
    for idx, row in df.head(5).iterrows():
        test_rows.append({
            'id': row['id'],
            'path': row['path'],
            'text_encoded': row.get('text_encoded', ''),
            **{c: row[c] for c in df.columns if c.startswith('label_')}
        })
    
    # Create test dataset
    try:
        test_ds = MultimodalStudyDataset(test_rows, n_slices=16, max_text_len=256)
        sample = test_ds[0]
        
        print("✓ Dataset loading successful!")
        print(f"\nSample batch shapes:")
        print(f"  Image: {sample['image'].shape}")
        print(f"  Text tokens: {sample['text'].shape}")
        if sample['label'] is not None:
            print(f"  Labels: {sample['label'].shape}")
        print(f"  ID: {sample['id']}")
        
        # Test batch collation
        test_loader = DataLoader(test_ds, batch_size=2, collate_fn=collate_multimodal)
        batch = next(iter(test_loader))
        print(f"\n✓ Batch collation successful!")
        print(f"  Batch image shape: {batch['image'].shape}")
        print(f"  Batch text shape: {batch['text'].shape}")
        if batch['label'] is not None:
            print(f"  Batch labels shape: {batch['label'].shape}")
    except Exception as e:
        print(f"✗ Error: {e}")
        import traceback
        traceback.print_exc()

## 7. Model Initialization and Validation

In [ ]:
# Initialize model
model = MidFusionMultimodal(
    n_outputs=12,
    image_backbone='tf_efficientnet_b3',
    embed_dim=256,
    vocab_size=5000,
    lr=1e-4
)

print("✓ Model initialized")
print(f"\nModel architecture:")
print(f"  Image Encoder: {type(model.image_encoder).__name__}")
print(f"  Text Encoder: {type(model.text_encoder).__name__}")
print(f"  Fusion Head: {type(model.fusion_head).__name__}")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Test forward pass if batch available
if csv_path.exists():
    try:
        with torch.no_grad():
            outputs = model(batch['image'], batch['text'])
        print(f"\n✓ Forward pass successful!")
        print(f"  Output shape: {outputs.shape}")
        print(f"  Output range: [{outputs.min().item():.3f}, {outputs.max().item():.3f}]")
    except Exception as e:
        print(f"✗ Forward pass error: {e}")
        import traceback
        traceback.print_exc()

## 8. End-to-End Training Example (Single Fold)

In [ ]:
def train_fold_multimodal(csv_path, fold=0, batch_size=8, max_epochs=20, 
                         log_dir='runs', checkpoint_dir='models'):
    """Train mid-fusion model on a single fold."""
    
    # Load CSV
    df = pd.read_csv(csv_path)
    
    # Split by fold
    train_rows = []
    val_rows = []
    
    for idx, row in df.iterrows():
        row_dict = {
            'id': row['id'],
            'path': row['path'],
            'text_encoded': row.get('text_encoded', ''),
            **{c: row[c] for c in df.columns if c.startswith('label_')}
        }
        
        if row.get('fold', -1) != fold:
            train_rows.append(row_dict)
        else:
            val_rows.append(row_dict)
    
    print(f"Fold {fold}: {len(train_rows)} train, {len(val_rows)} val")
    
    # Create datasets
    train_ds = MultimodalStudyDataset(train_rows, n_slices=16, max_text_len=256)
    val_ds = MultimodalStudyDataset(val_rows, n_slices=16, max_text_len=256)
    
    # Create dataloaders
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                            num_workers=4, collate_fn=collate_multimodal)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                          num_workers=4, collate_fn=collate_multimodal)
    
    # Initialize model
    model = MidFusionMultimodal(n_outputs=12, lr=1e-4)
    
    # Setup logging and checkpoints
    os.makedirs(log_dir, exist_ok=True)
    os.makedirs(checkpoint_dir, exist_ok=True)
    
    logger = TensorBoardLogger(save_dir=log_dir, name='mid_fusion', 
                              version=f'fold{fold}')
    checkpoint = ModelCheckpoint(
        dirpath=checkpoint_dir,
        filename=f'mid_fusion_fold{fold}-' + '{epoch}-{val/macro_auc:.4f}',
        monitor='val/macro_auc',
        mode='max',
        save_top_k=1
    )
    early_stop = EarlyStopping(
        monitor='val/macro_auc',
        mode='max',
        patience=7,
        verbose=True
    )
    
    # Train
    trainer = pl.Trainer(
        max_epochs=max_epochs,
        logger=logger,
        callbacks=[checkpoint, early_stop],
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        devices=1,
        precision='16-mixed' if torch.cuda.is_available() else '32',
        log_every_n_steps=10,
        num_sanity_val_steps=2
    )
    
    trainer.fit(model, train_loader, val_loader)
    
    # Save OOF predictions
    best_path = checkpoint.best_model_path
    if best_path:
        print(f"\nBest checkpoint: {best_path}")
        best_model = MidFusionMultimodal.load_from_checkpoint(best_path)
        preds = trainer.predict(best_model, dataloaders=val_loader)
        all_probs = np.concatenate(preds, axis=0)
        ids = [r['id'] for r in val_rows]
        
        # Save
        os.makedirs('preds', exist_ok=True)
        np.save(f'preds/mid_fusion_fold{fold}.npy', 
               {'ids': ids, 'probs': all_probs})
        print(f"Saved OOF predictions: preds/mid_fusion_fold{fold}.npy")
    
    return trainer, model


# Uncomment to train a fold
# if csv_path.exists():
#     trainer, model = train_fold_multimodal(csv_path, fold=0, batch_size=8, 
#                                           max_epochs=20)

## 9. Multi-Fold Training Wrapper

In [ ]:
def train_all_folds_multimodal(csv_path, n_folds=5, batch_size=8, max_epochs=20):
    """Train mid-fusion model on all folds and collect OOF predictions."""
    
    oof_results = []
    
    for fold in range(n_folds):
        print(f"\n{'='*60}")
        print(f"Training Fold {fold+1}/{n_folds}")
        print(f"{'='*60}")
        
        trainer, model = train_fold_multimodal(
            csv_path, fold=fold, batch_size=batch_size, 
            max_epochs=max_epochs
        )
        oof_results.append(fold)
    
    print(f"\n✓ All folds trained!")
    print(f"OOF predictions saved in preds/ directory")
    
    # Collect and ensemble
    print(f"\nCollecting OOF predictions...")
    all_ids = []
    all_probs = []
    
    for fold in range(n_folds):
        data = np.load(f'preds/mid_fusion_fold{fold}.npy', allow_pickle=True).item()
        all_ids.extend(data['ids'])
        all_probs.append(data['probs'])
    
    # Create OOF CSV
    oof_df = pd.DataFrame()
    oof_df['id'] = all_ids
    
    all_probs = np.vstack(all_probs)
    for i in range(all_probs.shape[1]):
        oof_df[f'pred_{i}'] = all_probs[:, i]
    
    oof_df.to_csv('challenges/kaggle-knee-mri/oof_mid_fusion.csv', index=False)
    print(f"✓ OOF predictions saved to: challenges/kaggle-knee-mri/oof_mid_fusion.csv")
    
    return oof_df

# Uncomment to run 5-fold training
# if csv_path.exists():
#     oof_df = train_all_folds_multimodal(csv_path, n_folds=5, batch_size=8, max_epochs=20)

## 10. Summary and Next Steps

### What This Notebook Does:
✅ Defines a mid-fusion multimodal architecture (image + text)  
✅ Integrates with preprocessed `.npz` files from `preprocess.py`  
✅ Validates dataset loading and batch formatting  
✅ Provides end-to-end training with PyTorch Lightning  
✅ Supports 5-fold cross-validation  
✅ Saves OOF predictions for ensembling  

### To Use This Notebook:
1. Ensure your CSV has columns: `id`, `path`, `text_encoded`, `fold`, `label_0`...`label_11`
2. Update the `csv_path` variable to point to your dataset CSV
3. Run validation cells to ensure data loading works
4. Uncomment and run the training cells
5. Collect OOF predictions for final ensemble

### Expected Performance:
- Macro AUC should improve over baseline (image-only) models
- Text features capture clinical context not visible in images
- Fusion at the embedding level preserves modality-specific information
